# Exercise Sheet 3

## Exercise 1: Shapley Values

In [1]:
import numpy as np
import math
import itertools

In [2]:
np.random.seed(123456)

## a)

In [3]:
def payoff(coalition):
    # Define boolean variables that indicate whether Timnit, Margret, Samy, Jeff and Larry are in the set or not.
    t = 't' in coalition
    s = 's' in coalition
    m = 'm' in coalition
    j = 'j' in coalition
    l = 'l' in coalition
    return 10*t + 10*m + 10*s + 2*j + 20 * (t and m) + 20 * (t and m and s) - 30 * ((t or m or s) and j)

In [4]:
# Testing
population = ['j', 'l', 'm', 's', 't']
print(payoff(('t','m')))
print(payoff(['t','j','s']))
print(payoff([]))
print(payoff(population))

40
-8
0
42


## b)

In [5]:
def all_unique_subsets(population):
    if not population:
        return [[]]
    else:
        population = list(set(population)) # remove double elements
        subsets_wo_member = all_unique_subsets(population[1:])
        subsets_w_member = [s + [population[0]] for s in subsets_wo_member]
        return subsets_w_member + subsets_wo_member

In [6]:
# Official solution: https://docs.python.org/2/library/itertools.html#recipes
# def all_unique_subsets(population):
#    population = list(set(population))
#    power_set = list(itertools.chain.from_iterable(itertools.combinations(population, r) for r in range(len(population)+1)))
#    return [list(subset) for subset in power_set]

In [7]:
# Testing
print(all_unique_subsets([]))
print(all_unique_subsets(['t']))
print(all_unique_subsets(population))

[[]]
[['t'], []]
[['t', 'm', 'j', 's', 'l'], ['m', 'j', 's', 'l'], ['t', 'j', 's', 'l'], ['j', 's', 'l'], ['t', 'm', 's', 'l'], ['m', 's', 'l'], ['t', 's', 'l'], ['s', 'l'], ['t', 'm', 'j', 'l'], ['m', 'j', 'l'], ['t', 'j', 'l'], ['j', 'l'], ['t', 'm', 'l'], ['m', 'l'], ['t', 'l'], ['l'], ['t', 'm', 'j', 's'], ['m', 'j', 's'], ['t', 'j', 's'], ['j', 's'], ['t', 'm', 's'], ['m', 's'], ['t', 's'], ['s'], ['t', 'm', 'j'], ['m', 'j'], ['t', 'j'], ['j'], ['t', 'm'], ['m'], ['t'], []]


In [8]:
def shapley_set(member, population, v_function=payoff, *args):
    population = list(set(population)) # remove double elements
    remainder = population[:] # copy
    remainder.remove(member)
    all_coalitions_without_member = all_unique_subsets(remainder)
    result = 0
    F = len(population)
    for coalition in all_coalitions_without_member:
        S = len(coalition)
        diff = v_function(coalition + [member], *args) - v_function(coalition, *args)
        factor = math.factorial(S) * math.factorial(F-S-1) / math.factorial(F)
        result += factor * diff
    return result

In [9]:
# Testing and Sanity Check
def all_shapley_values(population, shapley_fct, v_function=payoff, *args):
    return {member:shapley_fct(member, population, v_function, *args) for member in population}

def shapley_test(population, shapley_fct, v_function=payoff, *args):
    shapley_values=all_shapley_values(population, shapley_fct, v_function, *args)
    print("\n".join(f"Player {player}: {shapley_values[player]}" for player in shapley_values))
    sum_ = sum(shapley_values.values())
    print(sum_)
    print(v_function(population, *args))
    print(math.isclose(sum_, v_function(population, *args))) # efficiency axiom

In [10]:
shapley_test(population, shapley_set)

Player j: -20.5
Player l: 0.0
Player m: 24.166666666666668
Player s: 14.166666666666668
Player t: 24.166666666666668
42.0
42
True


### Definition of a few useful functions

In [11]:
def all_coalitions_without_members(members, population):
    population_without_members = population.copy()
    population_without_members.remove(members)
    return all_unique_subsets(population_without_members)

def payoff_diff(member, coalition, v_function=payoff, *args):
    return v_function(coalition + [member], *args) - v_function(coalition, *args)

def payoff_diff_list(member, coalitions, v_function=payoff, *args):
    return np.asarray([payoff_diff(member, coalition, v_function, *args) for coalition in coalitions])

In [12]:
def shapley_set_2(member, population, v_function=payoff, *args):
    population = list(set(population)) # remove double elements
    all_coalitions_without_member = all_coalitions_without_members(member, population)

    payoff_diffs = payoff_diff_list(member, all_coalitions_without_member, v_function, *args)

    F = len(population)
    weights = np.asarray([math.factorial(len(coalition)) * math.factorial(F - len(coalition) - 1) / math.factorial(F)
                          for coalition in all_coalitions_without_member])
    return np.sum(weights*payoff_diffs)

In [13]:
# Testing and Sanity Check
shapley_test(population, shapley_set_2)

Player j: -20.5
Player l: 0.0
Player m: 24.166666666666668
Player s: 14.166666666666666
Player t: 24.166666666666668
42.0
42
True


## c)

In [14]:
def shapley_perm(member, population, v_function=payoff, *args):
    # Note that itertools.permutations returns tuples, which need to be converted to lists first
    # permutation.index(member) returns the index of member in the permutation, therfore
    # permutation[:permutation.index(member)] is exactly the permutation before this member
    coalitions = [permutation[:permutation.index(member)] for permutation in itertools.permutations(population)]
    return np.mean([payoff_diff(member, list(coalition), v_function, *args) for coalition in coalitions])

In [15]:
# Testing and Sanity Check
shapley_test(population, shapley_perm)

Player j: -20.5
Player l: 0.0
Player m: 24.166666666666668
Player s: 14.166666666666666
Player t: 24.166666666666668
42.0
42
True


## d)

In [16]:
def shapley_perm_approx(member, population, v_function=payoff, *args, its=100):
    vals = []
    rng = np.random.default_rng()
    for i in range(its):
        permutation = list(rng.permutation(population))
        coalition = permutation[:permutation.index(member)]
        vals.append(payoff_diff(member, coalition, v_function, *args))
    return np.mean(vals)

In [17]:
# Testing
for its in [1000, 10000, 10**5, 10**6]:
    print(shapley_perm_approx('t', population, its=its))
shapley_test(population, shapley_perm_approx)
shapley_test(population, lambda *args: shapley_perm_approx(*args, its=100000))

24.28
23.986
24.1028
24.1676
Player j: -18.7
Player l: 0.0
Player m: 23.9
Player s: 14.8
Player t: 19.9
39.9
42
False
Player j: -20.518
Player l: 0.0
Player m: 24.1855
Player s: 14.0966
Player t: 24.2337
41.9978
42
False


Clear convergence is apparent, but it is not very fast, although with 100000 iterations we already get quite a good approximation.

The exact numbers here depend of course on the specific parameters, the seed etc ...

## e) 

### i) Symmetry Check

In [18]:
def symmetry_check(j, k, population, shapley_func, v_function, *args):
    remainder = set(population) - set([j, k])
    all_S = all_unique_subsets(remainder)
    surpluss_j = []
    surpluss_k = []
    for S in all_S:
        surplus_j = v_function(S + [j], *args) - v_function(S, *args)
        surplus_k = v_function(S + [k], *args) - v_function(S, *args)
        surpluss_j.append(surplus_j)
        surpluss_k.append(surplus_k)
    surpluss_j, surpluss_k = np.array(surpluss_j), np.array(surpluss_k)
    equal_surplus = np.all(surpluss_j == surpluss_k)
    if equal_surplus:
        print('equal surplus')
        val_j = shapley_func(j, population, v_function, *args)
        val_k = shapley_func(k, population, v_function, *args)
        return val_j == val_k
    else:
        return True


In [19]:
symmetry_check('m', 't', population, shapley_set_2, payoff)

equal surplus


np.True_

### ii) Dummy property check

In [20]:
def dummy_check(j, population, shapley_func, v_function, *args):
    remainder = set(population) - set([j])
    all_S = all_unique_subsets(remainder)
    surpluss_j = []
    for S in all_S:
        surplus_j = v_function(S + [j], *args) - v_function(S, *args)
        surpluss_j.append(surplus_j)
    has_contribution = np.sum(np.abs(surpluss_j)) > 0
    if has_contribution:
        print('has contribution')
        val_j = shapley_func(j, population, v_function, *args)
        return val_j > 0
    else:
        return True

In [21]:
dummy_check('l', population, shapley_set_2, payoff)

True

### iii) Additivity check

In [22]:
def additivity_check(population, shapley_func, v_function1, v_function2, *args):
    combined = lambda x : v_function1(x, *args) + v_function2(x, *args)
    vals1 = np.array([shapley_func(j, population, v_function1, *args) for j in population])
    vals2 = np.array([shapley_func(j, population, v_function2, *args) for j in population])
    vals_comb = np.array([shapley_func(j, population, combined) for j in population])
    vals_additive = vals1 + vals2
    return np.all(vals_comb == vals_additive)


In [23]:
payoff2 = payoff

additivity_check(population, shapley_set_2, payoff, payoff2)

np.True_

### iv): Efficiency check

In [24]:
def efficiency_check(population, shapley_func, v_function, *args):
    payoff_total = v_function(population, *args)
    shapley_vals = [shapley_func(j, population, v_function, *args) for j in population]
    total_shapley_vals = np.sum(shapley_vals)
    return math.isclose(payoff_total, total_shapley_vals)

In [25]:
efficiency_check(population, shapley_set_2, payoff)

np.True_

## Exercise 2: SHAP

In [26]:
!pip install shap

In [27]:
import math
import numpy as np
%matplotlib inline
import pandas as pd

In [28]:
np.random.seed(123456)


### a)

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv('fifa.csv')
X, y = df.loc[:, df.columns != 'Man of the Match'], df['Man of the Match']
y = (y == 'Yes')
X = X.select_dtypes([np.number]).copy()
X = X.dropna(axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

classifier_RF = RandomForestClassifier(random_state=123456)
classifier_RF.fit(X_train, y_train)


In [30]:
# Testing
print(X_test.shape)
print(classifier_RF.predict(X_test.iloc[1:2]))
print(classifier_RF.predict(X_test.iloc[1:10]))
print(classifier_RF.predict_proba(X_test.iloc[1:10]))
print(classifier_RF.predict_proba(X_test.iloc[1:10])[:, 1])

(39, 18)
[False]
[False False  True  True  True False False False False]
[[0.89 0.11]
 [0.7  0.3 ]
 [0.45 0.55]
 [0.23 0.77]
 [0.26 0.74]
 [0.56 0.44]
 [0.81 0.19]
 [0.63 0.37]
 [0.67 0.33]]
[0.11 0.3  0.55 0.77 0.74 0.44 0.19 0.37 0.33]


As a classifier, this model of course produces probabilities as outputs, which are then transformed into a binary prediction. This is not very well suited for Shapley values, since we basically require a numerical or continuous target, in particular for the efficiency axiom to hold up. We will therefore always consider the predicted probabilities in this exercise, although one could also use the predict() function everywhere in the following, which would then be averaged when calculating PDPs or Shapley values.

In [31]:
# Note: Here we predict probabilities, and a standard linear regression is used, not logistic regression.

predict_RF = lambda x : classifier_RF.predict_proba(x)[:, 1]

### b) marginal sampling based SHAP value function

In [32]:
def marginal_vfunc(S, observation, X, predict, nr_samples = 100):
    X_tmp = X.sample(nr_samples, replace=True).copy()
    # Next step is just reshaping and copying the observation nr_samples many times
    observation_df = pd.DataFrame(observation).T.sample(nr_samples, replace=True)
    observation_df = observation_df.reset_index(drop=True)
    X_tmp.loc[:, S] = np.array(observation_df.loc[:, S].copy())
    return np.mean(predict(X_tmp))

In [33]:
# Testing
print(marginal_vfunc(X_test.columns[0:3], X_test.iloc[1,], X_train, predict_RF, nr_samples = 1000))
print(marginal_vfunc(X_test.columns[0:3], X_test.iloc[2,], X_train, predict_RF, nr_samples = 1000))

0.23195999999999997
0.59027


### c) Using this value function in the approximation algo for Shapley values

In [34]:
# Testing the approximation algorithm NOTE: This takes a while
print(shapley_perm_approx(X_test.columns[2], X_test.columns, marginal_vfunc, X_test.iloc[2,], X_train, predict_RF))
shapley_test(X_test.columns, shapley_perm_approx, marginal_vfunc, X_test.iloc[2,], X_train, predict_RF)

-0.004519999999999993
Player Goal Scored: 0.084712
Player Ball Possession %: -0.075936
Player Attempts: 0.0020389999999999974
Player On-Target: 0.015519000000000012
Player Off-Target: -0.011884999999999993
Player Blocked: -0.014625999999999997
Player Corners: -0.013124000000000007
Player Offsides: 0.006826000000000002
Player Free Kicks: -0.020009000000000006
Player Saves: -0.046861999999999994
Player Pass Accuracy %: 0.016273999999999997
Player Passes: -0.071127
Player Distance Covered (Kms): -0.01255100000000001
Player Fouls Committed: -0.019574999999999995
Player Yellow Card: -0.029343999999999988
Player Yellow & Red: 0.001807000000000007
Player Red: 0.0016869999999999919
Player Goals in PSO: -0.0019100000000000146
-0.188085
0.3
False


The approximation seems not to be very good, but maybe this is to be expected, since we use a small number of iterations for both the value function (the PD-functions) and the Shapley value approximation.

### d)

In [37]:
from scipy.special import comb
from sklearn.linear_model import LinearRegression

def shap_weights(mask):
    p = mask.shape[1]
    zs = np.sum(mask, axis=1)
    numerator = (p - 1)
    denominator = comb(p, zs) * zs * (p - zs)
    weights = numerator / denominator
    weights[~np.isfinite(weights)] = 0  # Handle all-zero/all-one coalitions
    return weights
    
def replace_dataset(obs, X, nr_samples):
    X_new = np.array(X.sample(nr_samples, replace=True))
    obs_rep = np.repeat(np.array(obs).reshape(-1, 1), X_new.shape[0], axis=1).T
    mask = np.random.binomial(1, 0.5, np.prod(X_new.shape)).reshape(X_new.shape)
    row_ixs, col_ixs = np.where(mask)
    X_new[row_ixs, col_ixs] = obs_rep[row_ixs, col_ixs]
    X_new = pd.DataFrame(X_new, columns=X.columns)
    return X_new, mask

def shap_data(obs, X, nr_samples, predict):
    X_new, mask = replace_dataset(obs, X, nr_samples)
    weight = shap_weights(mask)
    pred = predict(X_new)
    return mask, pred, weight

def kernel_shap(obs, X, nr_samples, predict):
    mask, pred, weight = shap_data(obs, X, nr_samples, predict)
    lm = LinearRegression()
    lm.fit(mask, pred, weight)
    coef = lm.coef_
    coef = pd.Series(coef, index=X.columns)
    coef['intercept'] = lm.intercept_
    return coef

### e)

In [38]:
# Note: Here we predict probabilities, and a standard linear regression is used, not logistic regression.

shap_vals = kernel_shap(X_test.iloc[0, :], X, 1000, predict_RF)
print(shap_vals)

Goal Scored               0.044606
Ball Possession %        -0.100817
Attempts                 -0.084817
On-Target                -0.083357
Off-Target                0.065834
Blocked                  -0.059500
Corners                  -0.015549
Offsides                  0.047230
Free Kicks               -0.028717
Saves                    -0.063862
Pass Accuracy %          -0.035855
Passes                   -0.034569
Distance Covered (Kms)    0.023506
Fouls Committed          -0.007121
Yellow Card               0.087759
Yellow & Red             -0.028795
Red                      -0.037265
Goals in PSO              0.015073
intercept                 0.631136
dtype: float64


### f)

`SHAP` values are expensive to compute. `TreeSHAP` offers a more efficient implementation that exploits the structure of tree-based models. Advanced knowledge: A further advantage of TreeSHAP is that it resamples the variables such that the joint distribution is preserved.

In [39]:
import shap

explainer = shap.TreeExplainer(classifier_RF, feature_perturbation='tree_path_dependent')
shap_values = explainer(X_test)
shap.plots.waterfall(shap_values[0][:, 1])

Distance covered had a strong positive effect on the predicted probability, whereas the number of free kicks, the number of off-target shots and the number of goals had a negative effect. 

## Exercise 3: Lime

In [40]:
!pip install lime

In [41]:
import numpy as np
%matplotlib inline

In [42]:
np.random.seed(123456)

### a)

In [43]:
import lime.lime_tabular
import lime

explainer = lime.lime_tabular.LimeTabularExplainer(np.array(X_train), feature_names=X.columns, class_names=classifier_RF.classes_)

exp = explainer.explain_instance(np.array(X_test.iloc[0, :]), classifier_RF.predict_proba, num_features=5, top_labels=1,
                                 distance_metric='l2')
exp.show_in_notebook(show_table=True, show_all=False)

### b)

In [44]:
explainer = lime.lime_tabular.LimeTabularExplainer(np.array(X_train), feature_names=X.columns,
                                                   class_names=classifier_RF.classes_, kernel_width=0.1)

exp = explainer.explain_instance(np.array(X_test.iloc[0, :]), classifier_RF.predict_proba, num_features=5, top_labels=1,
                                 distance_metric='l1')
exp.show_in_notebook(show_table=True, show_all=False)

The results depend on the metric used to define the neighborhood.

### c)

Both SHAP and Lime rely on a linear model approximation of the model. For lime, the normal feature values are used, for SHAP a transformed distribution indicating coalition membership for a sample is relied upon. Lime weights the different samples according to their distance to the observation of interest. In contrast, SHAP weights them according to the shapley kernel weights (which simulate sampling random permutations).

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=465b8883-1966-407a-bc10-026a1a84551f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>